### Scrapping of flipkart

In [3]:
import requests
import time
import os
import re
import pandas as pd
from bs4 import BeautifulSoup

In [4]:
#Readers for request
HEADERS=({"User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36","Accept-Language":"en-US, en;q=0.5"})

In [24]:
Categories={
     # "Mouse":"https://www.amazon.in/s?k=mouse&crid=51UQH71YVGOE&sprefix=mouse%2Caps%2C1031&ref=nb_sb_noss_2",
    # "Storybooks":"https://www.amazon.in/s?k=story+books&ref=nb_sb_noss",
    # "BoardGames":"https://www.amazon.in/s?k=board+games&crid=2C6DGCSMH697R&sprefix=board+games%2Caps%2C607&ref=nb_sb_noss_2",
     # "Shoes":"https://www.amazon.in/s?k=shoes&crid=2C0Y2Y5FCZUXZ&sprefix=shoe%2Caps%2C600&ref=nb_sb_noss_2",
    # "Beauty":"https://www.amazon.in/s?k=Beauty+Products&crid=T3W2DDQ942ZX&sprefix=beauty+products%2Caps%2C458&ref=nb_sb_noss_2",
    # "Phones":"https://www.amazon.in/s?k=phone&crid=3VOZQYA6KUITC&sprefix=phon%2Caps%2C583&ref=nb_sb_noss_2",
    # "Bottle":"https://www.amazon.in/s?k=bottle&crid=20D3F1ZBVBZSU&sprefix=bottle%2Caps%2C505&ref=nb_sb_noss_2",
    # "Laptop":"https://www.amazon.in/s?k=Laptop&crid=3KJPSAJF2X7DT&sprefix=laptop%2Caps%2C495&ref=nb_sb_noss_2",
    # "Earphones":"https://www.amazon.in/s?k=Earphone&ref=nb_sb_noss"
    "Lipstick":"https://www.amazon.in/s?k=lipstick&crid=1JOFLHEIVYQ0R&sprefix=lipstick%2Caps%2C855&ref=nb_sb_noss_2"
    # "Watch":"https://www.amazon.in/s?k=watch+for+woman&crid=1ZZ1R50R5F2D9&sprefix=watch+%2Caps%2C613&ref=nb_sb_ss_mvt-t11-ranker_2_6"
}
amazon_products=[]

In [25]:
from requests_html import HTMLSession
from urllib.parse import urljoin
s=HTMLSession()
Base="https://www.amazon.in"
for category ,value in Categories.items():
    print(f"\n Scrapping category : {category}")
    main_url = value
    while main_url:
      r = s.get(main_url,headers=HEADERS)
      soup = BeautifulSoup(r.text, "html.parser")
      print("Current page:", main_url)
      products= soup.find_all("div",{"data-component-type":"s-search-result"})   
      for product in products:
             extract_link=product.select_one("a.a-link-normal")
             product_link=extract_link.get("href") if extract_link else None
             if not product_link:
                 continue
             product_url = urljoin(Base,product_link)
             # print(product_url)
             product_response = s.get(product_url, headers=HEADERS)
             print("Status:", product_response.status_code)
             print("Final URL:", product_response.url)

             if "captcha" in product_response.url.lower() or "validatecaptcha" in product_response.url.lower():
              print("BLOCKED BY CAPTCHA")
              break

             if "Robot Check" in product_response.text:
              print("ROBOT CHECK PAGE")
              break
             product_soup = BeautifulSoup(product_response.text, "html.parser") 
         
             ##TITLE###
             title_tag = product_soup.select_one("h1 span")
             title = title_tag.text.strip() if title_tag else None
             print(title)
          
             ##  Price tag
             price_tag=product.find("span",{"class":"a-price-whole"})
             price= price_tag.text.replace(",","") if price_tag else 0
             print(price)
          
             # Discount Price
             discount_tag = product.find( "span",string=lambda t: t and "% " in t )
             text = discount_tag.get_text(strip=True)if discount_tag else "0"  # "(65% off)"
             discount = int("".join(c for c in text if c.isdigit()))
             print(discount)
          
             ### Real Price
             mrp_tag = product_soup.select_one("span.a-price.a-text-price span.a-offscreen")
             mrp = mrp_tag.text.replace("M.R.P.:  ₹", "").replace("₹","").replace(",", "").strip() if mrp_tag else None
             MRP = mrp if mrp else 0
             print(MRP)
             ## DElivery
             delivery_tag = product_soup.select_one("#mir-layout-DELIVERY_BLOCK-slot-PRIMARY_DELIVERY_MESSAGE_LARGE span")
             delivery_text= delivery_tag.get_text(" ", strip= True) if delivery_tag else None
             if delivery_text:
                 delivery_text = delivery_text.replace("Details", "").strip()
                 if delivery_text.startswith("FREE delivery"):
                    delivery_price = "FREE"
                    delivery_date = delivery_text.replace("FREE delivery", "").strip(" .")
                 elif "delivery" in delivery_text.lower():
                    if "₹" in delivery_text:
                       delivery_price = delivery_text.split("delivery")[0].strip()
                    else:
                       delivery_price = None
                 delivery_date = delivery_text.split("delivery")[-1].strip(" .")
             print(delivery_date)
             print(delivery_price)
             ## Review
             review_count=product_soup.find("span",{"id":"acrCustomerReviewText"})
             review=review_count.text.replace("(","").replace(")","").replace(",","") if review_count else None
             print(review)
             ##Rating 
             rating_tag=product_soup.find("span",{"class":"a-size-small a-color-base"})
             rating=rating_tag.text.replace(" ","") if rating_tag else None
             print(rating)
             if title and price:
                               amazon_products.append({
                                   "Title": title,
                                   "Offer_price": price,
                                   "Discount":discount,
                                   "Real_price":MRP,
                                   "Rating": rating,
                                   "Review":review,
                                   "Delivery Charge":delivery_price,
                                   "Delivery Date":delivery_date,
                                   "Category": category,
                                   "Platform": "Amazon"
                               })
      page = soup.find("ul",{"class":"a-unordered-list a-horizontal s-unordered-list-accessibility"})
      if not page:
         break
      next_btn = page.find("a", class_="s-pagination-next")
      if next_btn and "href" in next_btn.attrs:
        main_url = Base + next_btn["href"]   
      else:
        break


    


 Scrapping category : Lipstick
Current page: https://www.amazon.in/s?k=lipstick&crid=1JOFLHEIVYQ0R&sprefix=lipstick%2Caps%2C855&ref=nb_sb_noss_2
Status: 200
Final URL: https://www.amazon.in/LOVETC-Perfect-High-Definition-Matte-Lipstick/dp/B0FQJY4533/ref=sr_1_1_sspa?crid=1JOFLHEIVYQ0R&dib=eyJ2IjoiMSJ9.UEw53yhaVFJXiA5LPFzTnPQHYxa6JLb6Ri17IQJ1h5ehaQ7TyjKBZY-eQPoduV1hPsQkWrKTisAcvtijltsOXroIUS3Kh79T_GFClsURhJSMcXcm0AI3X_x_a8fgYScihrR6dS1qeit9oQD5NXfRLfyS2Bxf9wc0aEoyd8cD0c4rrwAf1yh8nA150d61eizg94p72cM47tux3eKrwaTaXX9Yv6Dz4mzmXJFjFfCozpUQCS-yJnB3Ywxwd200FjPdLS8I68msNmW4UJAShZdPs2gkCGw-NJzO6su6_QEdTr0.1pSkT4RdMP5bBifANmm4I21Yxk_jqm6XCz8BrT3Ax4s&dib_tag=se&keywords=lipstick&qid=1771862839&sprefix=lipstick%2Caps%2C855&sr=8-1-spons&aref=7fDXrZpJJ4&sp_csd=d2lkZ2V0TmFtZT1zcF9hdGY&psc=1
Pout, Perfect, Etc High-Definition Matte Lipstick - Birthday Blush
945
10
22500.00
Thursday, 26 February
FREE
46
4.7
Status: 200
Final URL: https://www.amazon.in/LOVETC-Perfect-High-Definition-Matte-Lipstick/dp/B0F

In [26]:
import pandas as pd
df_1=pd.DataFrame(amazon_products)
df_1.to_csv("amazon.csv", index=False)

In [27]:
df=pd.read_csv("amazon.csv")
df2=pd.read_csv("amazon_products.csv")
Dataframe = pd.concat([df,df2],ignore_index=True) 
Dataframe.to_csv("amazon_products.csv",index=False)

In [28]:
Dataframe.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6383 entries, 0 to 6382
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Title            6383 non-null   object 
 1   Offer_price      6383 non-null   float64
 2   Discount         6383 non-null   object 
 3   Real_price       6383 non-null   float64
 4   Rating           6100 non-null   object 
 5   Review           5917 non-null   float64
 6   Delivery Charge  6381 non-null   object 
 7   Delivery Date    6383 non-null   object 
 8   Category         6383 non-null   object 
 9   Platform         6383 non-null   object 
dtypes: float64(3), object(7)
memory usage: 498.8+ KB


In [29]:
DF=pd.read_csv("amazon_products.csv")

In [30]:
DF["Category"].value_counts()

Category
Watch         1515
BoardGames    1149
Shoes          756
Beauty         472
Earphones      425
Mouse          423
Phones         408
Lipstick       384
Laptop         369
Storybooks     354
Bottle         128
Name: count, dtype: int64

In [22]:
Dataframe.duplicated().sum()

1896

In [23]:
df_1=df_1.dropna()

In [6]:
!pip install httpx selectolax

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
!pip install --upgrade pip

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----------------- ---------------------- 0.8/1.8 MB 1.9 MB/s eta 0:00:01
   ----------------------- ---------------- 1.0/1.8 MB 1.9 MB/s eta 0:00:01
   ----------------------------------- ---- 1.6/1.8 MB 2.0 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 2.0 MB/s  0:00:01



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: To modify pip, please run the following command:
C:\Program Files\Python312\python.exe -m pip install --upgrade pip
